<a href="https://colab.research.google.com/github/leeeunhyeok1/omok-agent/blob/main/notebooks/all24-training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train all 24 coefficients · updated search engine

Open in **Google Colab**, use a **CPU** runtime, and run cells in order. The engine uses
standard-library CPU search. Fresh initialization registers the current repository
source, including improved fallback selection and evaluator scan reuse.

**Default:** two hours of training in four 30-minute segments, one game worker,
a Drive backup before training and after every segment. The game cap can end a
segment earlier. Limits are fresh per segment; stopped work resumes in the same run.
A longer run does not guarantee improvement or a new champion.

- **New experiment:** choose `ACTION = "fresh"` and an unused `RUN_NAME`.
- **Continue on the same VM:** choose `ACTION = "resume"`, keep `RUN_NAME`.
- **After a Colab reset:** choose `ACTION = "restore"`, keep `RUN_NAME`; leave the
  archive name blank to use the latest complete backup for that run.
- Do not point this notebook at the old `all24-training-colab` experiment. Updated
  source requires a fresh experiment. Do not change source or settings during resume.

There are no separate engine test-suite cells. The optimizer retains its normal
candidate correctness gates. Independent champion testing is optional and disabled.

In [15]:
# 1. Session settings. Change these before running the remaining cells.
ACTION = "resume"  # @param ["fresh", "resume", "restore"]
RUN_NAME = "all24-search-v2"  # @param {type:"string"}
SOURCE_REF = "main"  # @param {type:"string"}
BACKUP_FOLDER = "omok-backups"  # @param {type:"string"}
RESTORE_ARCHIVE = ""  # @param {type:"string"}
SEGMENTS = 4  # @param {type:"integer"}
SECONDS_PER_SEGMENT = 1800  # @param {type:"integer"}
MAX_GAMES_PER_SEGMENT = 250  # @param {type:"integer"}
RUN_TRAINING = True  # @param {type:"boolean"}
RUN_CHAMPION_TEST = False  # @param {type:"boolean"}
TEST_SECONDS = 7200  # @param {type:"integer"}
TEST_LABEL = "champion-test-1"  # @param {type:"string"}

import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import signal
import subprocess
import sys
import tarfile
import tempfile
from datetime import datetime, timezone

assert ACTION in ("fresh", "resume", "restore")
for name in (RUN_NAME, TEST_LABEL):
    assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", name), "Use a simple name without slashes."
assert RUN_NAME != "all24-training-colab", "Choose a fresh name for the updated engine."
assert SEGMENTS > 0 and SECONDS_PER_SEGMENT > 0 and MAX_GAMES_PER_SEGMENT > 0
assert TEST_SECONDS > 0
REPO_URL = "https://github.com/leeeunhyeok1/omok-agent.git"
REPO = Path("/content/omok-agent")
RUN_DIR = REPO / ".scratch" / RUN_NAME
SESSION_DIR = REPO / ".scratch" / "colab-notebook"
SESSION_METADATA = SESSION_DIR / (RUN_NAME + ".json")
ACTIVE_JOB = None
print("Python:", sys.version)
print("Action:", ACTION, "| run:", RUN_NAME)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Action: resume | run: all24-search-v2


In [ ]:
# 2. Mount Drive and list complete backups.
from google.colab import drive
from IPython.display import display

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive") / BACKUP_FOLDER
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def complete_backups():
    return sorted(p for p in DRIVE_DIR.glob(RUN_NAME + "-*.tar.gz")
                  if Path(str(p) + ".sha256").is_file())

for path in complete_backups()[-10:]:
    print(path.name, f"{path.stat().st_size / 2**20:.1f} MiB")
print("Backup directory:", DRIVE_DIR)

In [ ]:
# 3. Read restore/resume metadata before selecting engine source.
restored_metadata = None
RESTORE_PATH = None
if ACTION == "restore":
    if RUN_DIR.exists():
        raise RuntimeError("The local run already exists. Choose resume, or a different RUN_NAME.")
    if RESTORE_ARCHIVE:
        RESTORE_PATH = DRIVE_DIR / RESTORE_ARCHIVE
    else:
        choices = complete_backups()
        if not choices:
            raise FileNotFoundError("No complete backup for this RUN_NAME in the chosen Drive folder.")
        RESTORE_PATH = choices[-1]
    checksum_path = Path(str(RESTORE_PATH) + ".sha256")
    expected = checksum_path.read_text().split()[0]
    if sha256_file(RESTORE_PATH) != expected:
        raise RuntimeError("Backup checksum mismatch; choose another intact backup.")
    with tarfile.open(RESTORE_PATH, "r:gz") as archive:
        stream = archive.extractfile("metadata.json")
        if stream is None:
            raise RuntimeError("Missing backup metadata.")
        restored_metadata = json.load(stream)
    if restored_metadata["run_name"] != RUN_NAME:
        raise RuntimeError("Backup belongs to a different RUN_NAME.")
elif ACTION == "resume":
    if not RUN_DIR.is_dir() or not SESSION_METADATA.is_file():
        raise RuntimeError("Local run/session metadata missing. Use restore after a VM reset.")
    restored_metadata = json.loads(SESSION_METADATA.read_text())
if restored_metadata and restored_metadata["python"] != sys.version:
    raise RuntimeError("Exact Python build differs from this checkpoint. Preserve the backup; "
                       "use a compatible runtime to resume, or start a fresh experiment with a new name. "
                       "Do not edit runtime strings or hashes.")
print("Source:", restored_metadata["git_commit"] if restored_metadata else SOURCE_REF)

In [ ]:
# 4. Clone full history and select source. Existing experiments are never reset.
def git(*args, capture=False):
    return subprocess.run(["git", "-C", str(REPO), *map(str, args)], check=True,
                          text=True, stdout=subprocess.PIPE if capture else None).stdout

if not (REPO / ".git").is_dir():
    if REPO.exists():
        raise RuntimeError("Repository path exists without Git metadata; use a clean Colab VM.")
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
if git("status", "--porcelain", "--untracked-files=no", capture=True).strip():
    raise RuntimeError("Tracked source has local edits. Preserve them before switching source.")
untracked = git("ls-files", "--others", "--exclude-standard", "--", "omok", "offline", "players", "configs", capture=True).splitlines()
if any(Path(name).suffix in (".py", ".json") for name in untracked):
    raise RuntimeError("Engine source has uncommitted files. Use clean registered source.")
if restored_metadata:
    desired = restored_metadata["git_commit"]
    if ACTION == "restore":
        # The bundle contains the exact source plus historical opponent Git objects.
        bundle_dir = Path(tempfile.mkdtemp(prefix="omok-source-", dir="/content"))
        with tarfile.open(RESTORE_PATH, "r:gz") as archive:
            archive.extract("source.bundle", bundle_dir, filter="data")
        git("fetch", bundle_dir / "source.bundle", "HEAD")
    else:
        git("cat-file", "-e", desired + "^{commit}")
else:
    git("fetch", "origin", SOURCE_REF)
    desired = git("rev-parse", "FETCH_HEAD", capture=True).strip()
current = git("rev-parse", "HEAD", capture=True).strip()
if current != desired and any(n == "omok" or n.startswith("omok.") or
                              n == "offline" or n.startswith("offline.") for n in sys.modules):
    raise RuntimeError("Restart the Colab session before changing already imported engine source.")
git("checkout", "--detach", desired)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
SESSION_DIR.mkdir(parents=True, exist_ok=True)
print("Pinned source:", git("rev-parse", "HEAD", capture=True).strip())

In [ ]:
# 5. Initialize once, resume locally, or restore the whole saved run.
if ACTION == "fresh":
    if RUN_DIR.exists():
        raise RuntimeError("Run already exists. Select resume instead of initializing again.")
    subprocess.run([sys.executable, "docs/colab-training/initialize-all24-colab.py",
                    "--output", str(RUN_DIR)], check=True)
elif ACTION == "restore":
    staging = Path(tempfile.mkdtemp(prefix="omok-restore-", dir="/content"))
    with tarfile.open(RESTORE_PATH, "r:gz") as archive:
        archive.extractall(staging, filter="data")
    if RUN_DIR.exists():
        raise RuntimeError("Restore refuses to overwrite an existing run.")
    RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(staging / "run"), str(RUN_DIR))
    print("Restored:", RESTORE_PATH.name)

CONDITIONS = SESSION_DIR / (RUN_NAME + "-conditions.json")
CONDITIONS.write_text(json.dumps({
    "power": "Colab managed VM; power settings unknown",
    "load": "No other notebook training jobs running; shared-host load unknown",
    "other_experiments_stopped": True
}, indent=2))
metadata = {
    "format": "omok-colab-backup-v1", "run_name": RUN_NAME,
    "git_commit": git("rev-parse", "HEAD", capture=True).strip(),
    "git_status": git("status", "--porcelain", "--untracked-files=no", capture=True),
    "python": sys.version, "repository": REPO_URL,
    "created_utc": datetime.now(timezone.utc).isoformat()
}
SESSION_METADATA.write_text(json.dumps(metadata, indent=2))
STOP_FILE = RUN_DIR / "NOTEBOOK_STOP"
DRIVE_STOP = DRIVE_DIR / (RUN_NAME + ".STOP")
print("Run ready:", RUN_DIR)
subprocess.run([sys.executable, "-m", "offline.optimizer", "status",
                "--output", str(RUN_DIR)], check=True)

In [ ]:
# 6. Backup and command helpers. Backups are taken only after the controller stops.
def ensure_idle():
    if ACTIVE_JOB is not None and ACTIVE_JOB.poll() is None:
        raise RuntimeError("Wait for the active controller before backing up or testing.")

def backup_run():
    ensure_idle()
    from offline.batches import owner
    with owner(RUN_DIR):
        if git("status", "--porcelain", "--untracked-files=no", capture=True).strip():
            raise RuntimeError("Source changed during this experiment; preserve it separately first.")
        if git("rev-parse", "HEAD", capture=True).strip() != metadata["git_commit"]:
            raise RuntimeError("Source commit changed during this experiment.")
        stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
        name = RUN_NAME + "-" + stamp + ".tar.gz"
        with tempfile.TemporaryDirectory(prefix="omok-backup-", dir="/content") as directory:
            staging = Path(directory)
            saved_metadata = dict(metadata, backed_up_utc=datetime.now(timezone.utc).isoformat())
            (staging / "metadata.json").write_text(json.dumps(saved_metadata, indent=2))
            bundle = staging / "source.bundle"
            git("bundle", "create", bundle, "HEAD", "--all")
            local_archive = staging / name
            with tarfile.open(local_archive, "w:gz") as archive:
                archive.add(RUN_DIR, arcname="run")
                archive.add(staging / "metadata.json", arcname="metadata.json")
                archive.add(bundle, arcname="source.bundle")
                archive.add(CONDITIONS, arcname="conditions.json")
                archive.add(REPO / "docs/colab-training/all24-colab-setup.json", arcname="recipe.json")
            digest = sha256_file(local_archive)
            destination = DRIVE_DIR / name
            uploading = DRIVE_DIR / (name + ".uploading")
            shutil.copy2(local_archive, uploading)
            if sha256_file(uploading) != digest:
                raise RuntimeError("Drive copy checksum mismatch; local run is preserved.")
            uploading.replace(destination)
            checksum = Path(str(destination) + ".sha256")
            temporary_checksum = Path(str(checksum) + ".uploading")
            temporary_checksum.write_text(digest + "  " + name + "\n")
            temporary_checksum.replace(checksum)
        print("Drive backup:", destination, f"({destination.stat().st_size / 2**20:.1f} MiB)")
        return destination

def run_command(command, stop_file):
    global ACTIVE_JOB
    ensure_idle()
    ACTIVE_JOB = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, text=True, bufsize=1,
                                  start_new_session=True)
    job = ACTIVE_JOB
    try:
        for line in job.stdout:
            print(line, end="", flush=True)
        result = job.wait()
        if result:
            raise subprocess.CalledProcessError(result, command)
    except KeyboardInterrupt:
        stop_file.touch()
        job.send_signal(signal.SIGINT)
        print("Stopping dispatch and waiting for checkpoint/worker cleanup...")
        try:
            job.wait(timeout=120)
        except subprocess.TimeoutExpired:
            raise RuntimeError("Controller has not finished cleanup. No backup will be made while it runs.")
        raise
    finally:
        if job.poll() is not None:
            ACTIVE_JOB = None
        if ACTIVE_JOB is None and job.stdout is not None:
            job.stdout.close()

backup_run()  # Save the initialized/restored state before starting training.

## Train and back up automatically

Run the next cell. Each segment resumes the same saved optimizer and backs up after
it stops. The default dispatch budget is 4 × 30 minutes; cleanup and backup add time.
Game counters count attempts, not wins. A persisted promotion decision can still
retain the initial champion.

To stop, interrupt this cell once and let cleanup and backup finish. Alternatively,
create an empty file named `<RUN_NAME>.STOP` in the Drive backup folder; it stops
before the next segment, after the current segment and backup finish. To continue,
remove that Drive file and any local `NOTEBOOK_STOP` file, then rerun the training
cell. Do not start another controller for the same run.

Backups protect completed segments. A VM loss during a segment can lose work since
the preceding backup. A CPU runtime change can change clock-dependent play; exact
Python build and source compatibility are required for checkpoint resume.

In [ ]:
# 7. Bounded training. Rerun this cell to continue another set of segments.
if not RUN_TRAINING:
    print("Training disabled in session settings.")
else:
    for segment in range(SEGMENTS):
       # if DRIVE_STOP.exists() or STOP_FILE.exists():
            #print("Stop file found. Remove it when you want to continue.")
           # break
        print(f"\nSegment {segment + 1}/{SEGMENTS}: up to {SECONDS_PER_SEGMENT}s, "
              f"{MAX_GAMES_PER_SEGMENT} game attempts, one worker", flush=True)
        try:
            run_command([
                sys.executable, "-u", "-m", "offline.optimizer", "run",
                "--output", str(RUN_DIR), "--max-games", str(MAX_GAMES_PER_SEGMENT),
                "--max-seconds", str(SECONDS_PER_SEGMENT), "--workers", "1",
                "--conditions", str(CONDITIONS), "--stop-file", str(STOP_FILE),
                "--contention", "Colab CPU; one game worker; other notebook jobs stopped; shared-host load unknown"
            ], STOP_FILE)
        finally:
            if ACTIVE_JOB is None:
                backup_run()
        from offline.evolution import status
        current = status(RUN_DIR)
        print("Champion:", current["incumbent"]["identity"])
        print("Rounds:", [(r["index"], r["stage"]) for r in current["rounds"]])
        if current["stopped"]:
            print("Optimizer stop:", current["stopped"])
            break


Segment 1/4: up to 1800s, 250 game attempts, one worker


In [ ]:
# 8. Read current status and the latest committed-round review. No games are played.
ensure_idle()
from offline.evolution import status
current = status(RUN_DIR)
display({"champion": current["incumbent"]["identity"], "stopped": current["stopped"],
         "rounds": [{"index": r["index"], "stage": r["stage"], "decision": r["decision"]}
                    for r in current["rounds"]]})
committed = [r for r in current["rounds"] if r["stage"] == "committed"]
if committed:
    subprocess.run([sys.executable, "-m", "offline.optimizer", "review",
                    "--output", str(RUN_DIR), "--round", str(committed[-1]["index"])], check=True)
else:
    print("No committed round yet. Rerun the training cell to continue saved work.")

In [ ]:
# 9. Export the current champion and make an additional manual backup.
ensure_idle()
from offline.champions import ChampionStore
export_version = ChampionStore(RUN_DIR / "champions").load()["current"]
EXPORT = RUN_DIR / "notebook-exports" / (export_version + ".json")
subprocess.run([sys.executable, "-m", "offline.optimizer", "export",
                "--output", str(RUN_DIR), "--destination", str(EXPORT)], check=True)
backup_run()
print("Champion export included in the Drive archive:", EXPORT)

## Optional: independently test the saved champion

Leave `RUN_CHAMPION_TEST = False` to skip all cells below during Run all. Enable it
only after training has stopped. A new test freezes the exact retained champion;
if nobody was promoted, it tests the initial configuration. Reusing `TEST_LABEL`
resumes its saved plan, even if a later training session changes the champion.
Choose a new label to test a different version.

This uses the existing experiment APIs: 40 official-format development games,
eight per opponent, four per color, 60,000 ms total per player, one worker. This is
not final grading or a reserved holdout evaluation. No final holdout is accessed.
Results, incomplete games, and failures remain explicit. Tests can take longer
than their dispatch bound because of cleanup. The test directory is inside the
run and is included in subsequent Drive backups.

In [ ]:
# 10. Create or reuse a champion test plan. Disabled by default.
if RUN_CHAMPION_TEST:
    ensure_idle()
    from offline.champions import ChampionStore
    from offline.registry import Entry, default_catalog
    from offline.experiments import ExperimentPlan
    TEST_ROOT = RUN_DIR / "notebook-tests" / TEST_LABEL
    TEST_ROOT.mkdir(parents=True, exist_ok=True)
    TEST_PLAN = TEST_ROOT / "plan.json"
    TEST_RESULTS = TEST_ROOT / "results"
    if TEST_PLAN.exists():
        test_plan = ExperimentPlan.load(TEST_PLAN)
        print("Reusing saved test plan:", test_plan.identity)
    else:
        saved = ChampionStore(RUN_DIR / "champions").load()
        bundle = saved["versions"][saved["current"]]["bundle"]
        candidate = Entry.archived(bundle)
        _, opponents = default_catalog()
        test_plan = ExperimentPlan.resolve(candidate, opponents, purpose="training", mode="official",
            opponents=["simple", "random", "synthetic-aggressive", "synthetic-capture", "synthetic-search1"],
            move_cap=10000, diagnostics=True)
        test_plan.save(TEST_PLAN)
        print("Frozen champion:", candidate.identity)
    backup_run()  # Preserve the exact test plan before execution.
else:
    print("Optional champion testing disabled.")

In [ ]:
# 11. Run/resume the optional test, then back up its complete evidence.
if RUN_CHAMPION_TEST:
    TEST_STOP = TEST_ROOT / "STOP"
    if TEST_STOP.exists():
        raise RuntimeError("Remove the test STOP file when you want to continue this test.")
    command = [sys.executable, "-u", "-m", "offline.experiment_cli", "run",
               "--plan", str(TEST_PLAN), "--output", str(TEST_RESULTS),
               "--workers", "1", "--max-seconds", str(TEST_SECONDS),
               "--stop-file", str(TEST_STOP),
               "--contention", "Colab CPU; training stopped; one game worker; shared-host load unknown"]
    if (TEST_RESULTS / "checkpoint.json").exists():
        command.append("--resume")
    try:
        run_command(command, TEST_STOP)
    finally:
        if ACTIVE_JOB is None:
            backup_run()

In [ ]:
# 12. Display test scores and offer a champion download. No extra games.
if RUN_CHAMPION_TEST:
    from offline.matches import score_experiment
    raw = json.loads((TEST_RESULTS / "results.json").read_text())
    scores = score_experiment(ExperimentPlan.load(TEST_PLAN), raw)
    (TEST_ROOT / "scores.json").write_text(json.dumps(scores, indent=2))
    names = {job["opponent"]["identity"]: job["opponent"]["logical_id"]
             for job in test_plan.data["jobs"]}
    for match in scores["matches"]:
        display({"opponent": names.get(match["opponent_identity"], match["opponent_identity"]),
                 "counts": match["counts"], "score": match["recognized"],
                 "qualification": match["qualification"]})
    backup_run()
# Optional local file download; Drive already contains the export in the archive.
DOWNLOAD_CHAMPION = False
if DOWNLOAD_CHAMPION:
    from google.colab import files
    files.download(str(EXPORT))

## Resume checklist

1. On the same VM, choose `resume`, keep the run name, and rerun from the top.
2. On a new VM, choose `restore`, keep the run name and Drive folder, then run from
   the top. Blank `RESTORE_ARCHIVE` selects the latest checksum-backed archive.
3. Restore checks the checksum, Python build, and source commit. The optimizer
   also verifies its checkpoint and source identities before playing games.
4. If the Python runtime is incompatible, keep the backup intact. Start a new
   experiment under a new name if you cannot obtain the matching runtime.
5. Old backups are retained; this notebook never deletes them automatically.